In [1]:
# =============================================================================
# Cell 1 - bootstrap. Loads the prepared frame committed by nb31.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd
ALPHA=config.ALPHA_PRIMARY
PREPARED=config.PROC_DIR/'ciciot2023_prepared.parquet'
iot=pd.read_parquet(PREPARED)
rec=json.loads((config.REPORTS_DIR/'focal_class_record_ciciot2023.json').read_text())
FOCAL=rec['focal_class']; LABEL_COL=rec['label_column']
FEATS=json.loads((config.REPORTS_DIR/'ciciot2023_prepared_fingerprint.json').read_text())['features']
print('frame:', iot.shape, '| focal:', FOCAL, '| features:', len(FEATS))
print('focal subtypes:', rec['focal_subtypes'])


Mounted at /content/drive
frame: (1510142, 49) | focal: Web | features: 44
focal subtypes: ['Backdoor_Malware', 'BrowserHijacking', 'CommandInjection', 'SqlInjection', 'Uploading_Attack', 'XSS']


In [2]:
# =============================================================================
# Cell 2 - SOURCE / TARGET SPLIT. Preregistration section 4 draws train, val,
# probcal and the source calibration pool from the SOURCE, and D_eval and T_cal
# from the TARGET. NSL-KDD gets that split for free (KDDTrain+ / KDDTest+),
# CIC-IDS2017 from within-day session blocks and UGR'16 from July / August.
# CIC-IoT-2023 has no such structure in the CSV release, so the split is
# constructed here: a disjoint 70/30 row split, with the ladder then withholding
# focal-class subtypes from the SOURCE side so the target carries genuine
# support shift, exactly as KDDTest+ carries R2L subtypes absent from KDDTrain+.
#
# The partition committed by nb31 covered only the four source partitions; this
# supersedes it by reserving a target side first. The focal class is RE-DERIVED
# under the new split and must still be Web, or the gate is void.
# =============================================================================
SPLIT_SEED = 20260727
TARGET_FRAC = 0.30
rng = np.random.default_rng(SPLIT_SEED)

# stratify the source/target split by specific label so every subtype exists on both sides
side = pd.Series(index=iot.index, dtype=object)
for lbl, g in iot.groupby(LABEL_COL, sort=True):
    idx = g.index.to_numpy().copy(); rng.shuffle(idx)
    n_t = int(round(TARGET_FRAC*len(idx)))
    side.loc[idx[:n_t]] = 'target'; side.loc[idx[n_t:]] = 'source'
iot['side'] = side.values
print('side sizes:'); print(iot['side'].value_counts().to_string())
print('\nfocal rows by side:')
print(iot[iot.family==FOCAL].groupby(['side','subtype']).size().unstack('side').fillna(0).astype(int).to_string())

# four source partitions within the source side (section 4 proportions)
def strat(df, fr, seed, col):
    r=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float)
    big=nm[int(np.argmax(ff))]; a=pd.Series(index=df.index, dtype=object)
    for _, s in df.groupby(col, sort=True):
        idx=s.index.to_numpy().copy(); r.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)] += n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a
src = iot[iot.side=='source']
iot['partition'] = pd.Series(index=iot.index, dtype=object)   # nb31 persisted before partitioning
iot.loc[src.index,'partition'] = strat(src, config.SPLIT_FRACTIONS, SPLIT_SEED, LABEL_COL).values
iot.loc[iot.side=='target','partition'] = 'target_pool'
assert iot['partition'].notna().all(), 'unassigned rows after partitioning'
print('\npartitions:'); print(iot['partition'].value_counts().to_string())

# RE-VERIFY the gate under the new split
pool = iot[iot.partition=='source_cal_pool']; need = conformal.min_calib_n(ALPHA)
chk=[]
for f,g in iot.groupby('family'):
    n=int((pool['family']==f).sum())
    chk.append({'family':f,'n_source_cal_pool':n,'feasible':n>=need,
                'n_subtypes':int(g['subtype'].nunique()),
                'ladder_capable':bool(n>=need and g['subtype'].nunique()>=2)})
chk=pd.DataFrame(chk).sort_values('n_source_cal_pool')
print('\nfeasibility under the source/target split:'); print(chk.to_string(index=False))
cand=chk[(chk.family!='Benign')&(chk.ladder_capable)].sort_values('n_source_cal_pool')
assert len(cand)>0, 'no attack family is feasible and multi-subtype under the source/target split'
refocal=cand.iloc[0]['family']
print(f'\nre-derived focal class: {refocal}  (recorded: {FOCAL})')
assert refocal==FOCAL, f'focal class changed under the source/target split ({FOCAL} -> {refocal}); the gate must be re-recorded before proceeding'
print('GATE HOLDS: focal class unchanged')


side sizes:
side
source    1057097
target     453045

focal rows by side:
side              source  target
subtype                         
Backdoor_Malware    2253     965
BrowserHijacking    4101    1758
CommandInjection    3786    1623
SqlInjection        3671    1574
Uploading_Attack     876     376
XSS                 2692    1154

partitions:
partition
train              634309
target_pool        453045
source_cal_pool    158547
probcal            158547
val                105694

feasibility under the source/target split:
    family  n_source_cal_pool  feasible  n_subtypes  ladder_capable
BruteForce               1371      True           1           False
       Web               2603      True           6            True
    Benign               6299      True           1           False
  Spoofing              12575      True           2            True
     Mirai              18953      True           3            True
     Recon              23055      True           5      

In [3]:
# =============================================================================
# Cell 3 - LADDER. Variant holdout within the focal family, mirroring the NSL-KDD
# unseen-subtype ladder (Amendment 10 A10.5). At rung f, a fraction f of the
# focal class's evaluation mass is drawn from subtypes WITHHELD from the source
# side, so the family stays feasible under SHC while its composition shifts.
# Subtypes are chosen at random per realization, never ordered by frequency.
# =============================================================================
RUNGS = [0.00, 0.20, 0.40, 0.60, 0.80]
R_REAL = config.N_LADDER_REALIZATIONS['ciciot2023']
EVAL_N = 60000            # rows per D_eval, natural family prevalence
FOCAL_SUBS = sorted(iot[iot.family==FOCAL]['subtype'].unique())
N_HOLDOUT = 2             # subtypes withheld from source per realization
print(f'focal subtypes ({len(FOCAL_SUBS)}): {FOCAL_SUBS}')
print(f'rungs {RUNGS} | realizations {R_REAL} | holdout {N_HOLDOUT} subtypes | D_eval {EVAL_N} rows')

tgt = iot[iot.side=='target']
fam_prev = iot['family'].value_counts(normalize=True)
focal_quota = int(round(EVAL_N*fam_prev[FOCAL]))
print(f'\nfocal prevalence {fam_prev[FOCAL]:.4%} -> ~{focal_quota} focal rows per D_eval')
assert focal_quota >= 200, f'only {focal_quota} focal rows per eval; raise EVAL_N'

assign=[]      # list of per-rung DataFrames, concatenated once at the end
for j in range(R_REAL):
    r = np.random.default_rng(int(hashlib.sha256(f'iotladder|{j}'.encode()).hexdigest(),16)%(2**32))
    held = sorted(r.choice(FOCAL_SUBS, N_HOLDOUT, replace=False).tolist())
    seen = [s for s in FOCAL_SUBS if s not in held]
    tgt_held = tgt[(tgt.family==FOCAL)&(tgt.subtype.isin(held))].index.to_numpy()
    tgt_seen = tgt[(tgt.family==FOCAL)&(tgt.subtype.isin(seen))].index.to_numpy()
    tgt_other= tgt[tgt.family!=FOCAL].index.to_numpy()
    for rung in RUNGS:
        n_held = int(round(rung*focal_quota)); n_seen = focal_quota-n_held
        if n_held>len(tgt_held) or n_seen>len(tgt_seen):
            print(f'  realization {j} rung {rung}: quota unmet (need {n_held} held/{n_seen} seen, '
                  f'have {len(tgt_held)}/{len(tgt_seen)}) -> skipped')
            continue
        rr=np.random.default_rng(int(hashlib.sha256(f'iotdraw|{j}|{rung}'.encode()).hexdigest(),16)%(2**32))
        pick = np.concatenate([rr.choice(tgt_held,n_held,replace=False),
                               rr.choice(tgt_seen,n_seen,replace=False),
                               rr.choice(tgt_other,EVAL_N-focal_quota,replace=False)])
        # column-wise, not 60k dict appends per rung
        assign.append(pd.DataFrame({'realization':np.full(len(pick),j,dtype=np.int16),
                                    'rung':np.full(len(pick),rung,dtype=np.float64),
                                    'row_idx':pick.astype(np.int64),
                                    'held_out':'|'.join(held)}))
lad=pd.concat(assign, ignore_index=True)
print(f'\nladder rows: {len(lad):,} | realizations x rungs: {lad.groupby(["realization","rung"]).ngroups}')
comp = lad.merge(iot[['family','subtype']], left_on='row_idx', right_index=True)
comp = comp[comp.family==FOCAL].copy()
comp['is_held'] = [s in h.split('|') for s,h in zip(comp['subtype'], comp['held_out'])]
chk2 = comp.groupby('rung')['is_held'].mean()
print('\nrealised unseen-subtype fraction of focal eval mass, by rung (target = the rung value):')
print(chk2.round(4).to_string())
# Compare explicitly per rung. Do NOT subtract two Series and rely on index alignment:
# a float dtype mismatch there yields all-NaN, and .max() skips NaN, so the assertion
# would pass unconditionally even for a completely broken ladder.
devs = {}
for r in RUNGS:
    hits = [v for k, v in chk2.items() if abs(float(k) - r) < 1e-6]
    assert len(hits) == 1, f'rung {r} missing or duplicated in the realised ladder'
    devs[r] = abs(hits[0] - r)
dev = max(devs.values())
print('per-rung deviation:', {k: round(v, 4) for k, v in devs.items()})
print(f'max deviation from the requested rung: {dev:.4f}')
assert dev < 0.02, f'realised unseen fraction does not track the requested rung (max dev {dev:.4f})'


focal subtypes (6): ['Backdoor_Malware', 'BrowserHijacking', 'CommandInjection', 'SqlInjection', 'Uploading_Attack', 'XSS']
rungs [0.0, 0.2, 0.4, 0.6, 0.8] | realizations 5 | holdout 2 subtypes | D_eval 60000 rows

focal prevalence 1.6442% -> ~986 focal rows per D_eval

ladder rows: 1,500,000 | realizations x rungs: 25

realised unseen-subtype fraction of focal eval mass, by rung (target = the rung value):
rung
0.0    0.0000
0.2    0.1998
0.4    0.3996
0.6    0.6004
0.8    0.8002
per-rung deviation: {0.0: 0.0, 0.2: 0.0002, 0.4: 0.0004, 0.6: 0.0004, 0.8: 0.0002}
max deviation from the requested rung: 0.0004


In [4]:
# =============================================================================
# Cell 4 - SHIFT MEASUREMENT (section 9). S_cov by cross-fitted domain classifier,
# S_lab as TV distance of family priors, S_sup as focal eval mass in subtypes
# absent from the source side. Measured per realization-rung, never inferred from
# the rung index.
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
SUB = 20000     # section 9: fixed subsample per side

srcpool_idx = iot[iot.partition=='source_cal_pool'].index.to_numpy()
rows=[]
for (j,rung), g in lad.groupby(['realization','rung']):
    ev = g['row_idx'].to_numpy(); held = set(g['held_out'].iloc[0].split('|'))
    rs = np.random.default_rng(int(hashlib.sha256(f'iotshift|{j}|{rung}'.encode()).hexdigest(),16)%(2**32))
    a = rs.choice(srcpool_idx, min(SUB,len(srcpool_idx)), replace=False)
    b = rs.choice(ev,          min(SUB,len(ev)),          replace=False)
    X = np.vstack([iot.loc[a,FEATS].to_numpy(float), iot.loc[b,FEATS].to_numpy(float)])
    y = np.r_[np.zeros(len(a)), np.ones(len(b))]
    aucs=[]
    for tr,te in StratifiedKFold(5, shuffle=True, random_state=0).split(X,y):
        m=HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(X[tr],y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:,1]))
    s_cov=float(np.mean(aucs))
    pa=iot.loc[a,'family'].value_counts(normalize=True)
    pb=iot.loc[b,'family'].value_counts(normalize=True)
    fams=sorted(set(pa.index)|set(pb.index))
    s_lab=float(0.5*np.abs(np.array([pa.get(f,0) for f in fams])-np.array([pb.get(f,0) for f in fams])).sum())
    fe=iot.loc[ev]; fe=fe[fe.family==FOCAL]
    s_sup=float(fe['subtype'].isin(held).mean()) if len(fe) else 0.0
    rows.append({'realization':j,'rung':rung,'S_cov':round(s_cov,4),
                 'S_lab':round(s_lab,4),'S_sup':round(s_sup,4),'held_out':'|'.join(sorted(held))})
    print(f'  r{j} rung {rung:.2f}: S_cov={s_cov:.4f} S_lab={s_lab:.4f} S_sup={s_sup:.4f}')
shift=pd.DataFrame(rows)
print('\nshift by rung (mean over realizations):')
print(shift.groupby('rung')[['S_cov','S_lab','S_sup']].mean().round(4).to_string())
print(f'\nS_cov range {shift.S_cov.min():.4f} - {shift.S_cov.max():.4f} '
      f'| S_sup range {shift.S_sup.min():.4f} - {shift.S_sup.max():.4f}')


  r0 rung 0.00: S_cov=0.4984 S_lab=0.0099 S_sup=0.0000
  r0 rung 0.20: S_cov=0.5018 S_lab=0.0052 S_sup=0.1998
  r0 rung 0.40: S_cov=0.4997 S_lab=0.0102 S_sup=0.3996
  r0 rung 0.60: S_cov=0.5008 S_lab=0.0081 S_sup=0.6004
  r0 rung 0.80: S_cov=0.5040 S_lab=0.0079 S_sup=0.8002
  r1 rung 0.00: S_cov=0.5053 S_lab=0.0061 S_sup=0.0000
  r1 rung 0.20: S_cov=0.5010 S_lab=0.0179 S_sup=0.1998
  r1 rung 0.40: S_cov=0.5005 S_lab=0.0069 S_sup=0.3996
  r1 rung 0.60: S_cov=0.5024 S_lab=0.0133 S_sup=0.6004
  r1 rung 0.80: S_cov=0.5042 S_lab=0.0090 S_sup=0.8002
  r2 rung 0.00: S_cov=0.5001 S_lab=0.0053 S_sup=0.0000
  r2 rung 0.20: S_cov=0.5051 S_lab=0.0057 S_sup=0.1998
  r2 rung 0.40: S_cov=0.4982 S_lab=0.0047 S_sup=0.3996
  r2 rung 0.60: S_cov=0.4945 S_lab=0.0075 S_sup=0.6004
  r2 rung 0.80: S_cov=0.5006 S_lab=0.0129 S_sup=0.8002
  r3 rung 0.00: S_cov=0.5017 S_lab=0.0112 S_sup=0.0000
  r3 rung 0.20: S_cov=0.4942 S_lab=0.0065 S_sup=0.1998
  r3 rung 0.40: S_cov=0.5002 S_lab=0.0071 S_sup=0.3996
  r3 rung 

In [ ]:
# =============================================================================
# Cell 5 - persist and commit. No coverage computed here.
# =============================================================================
iot[['side','partition']].to_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
lad.to_parquet(config.PROC_DIR/'ciciot2023_ladder_assignments.parquet', index=False)
shift.to_csv(config.REPORTS_DIR/'ladder_shift_measures_ciciot2023.csv', index=False)
chk.to_csv(config.REPORTS_DIR/'feasibility_after_split_ciciot2023.csv', index=False)
(config.REPORTS_DIR/'ciciot2023_split_record.json').write_text(json.dumps({
  'split_seed':SPLIT_SEED,'target_frac':TARGET_FRAC,'rungs':RUNGS,'realizations':R_REAL,
  'eval_n':EVAL_N,'n_holdout_subtypes':N_HOLDOUT,'focal_class':FOCAL,
  'focal_subtypes':FOCAL_SUBS,'focal_quota_per_eval':focal_quota,
  'gate_reverified':True,
  'note':'nb31 partitioned the whole frame into the four source partitions only. This '
         'notebook supersedes that by reserving a 30 per cent target side first, as '
         'preregistration section 4 requires, and re-derives the focal class under the new '
         'split as a check. Ladder is variant holdout within the focal family (Amendment 10 A10.5).'
}, indent=2, default=str))
print('saved split, ladder, shift measures')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb32: CIC-IoT-2023 source/target split, variant-holdout ladder, measured shift (no coverage)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved split, ladder, shift measures
